# Notebook 01: Tools in Isolation

**Core message:** Tools are plain Python functions. There is no magic.

We'll inspect the source, the JSON schema, call each tool manually, and see how error handling works.

In [ ]:
import sys, os
from pathlib import Path
# Move to project root so relative data paths resolve correctly
_here = Path(os.getcwd())
if not (_here / 'data' / 'scenes').exists() and (_here.parent / 'data' / 'scenes').exists():
    os.chdir(_here.parent)
sys.path.insert(0, str(Path(os.getcwd())))

import inspect
from agent.scene import load_scene
from agent.types import BoundingBox
from agent import tools
from agent.schemas import TOOL_SCHEMAS

load_scene('scene_a')
print('Scene A loaded.')

## One Tool in Detail: `compute_ndvi`

First, let's read the source:

In [ ]:
print(inspect.getsource(tools.compute_ndvi))

In [ ]:
# Call it directly — no agent, no API
bb = BoundingBox(row_min=0, row_max=200, col_min=0, col_max=200)
result = tools.compute_ndvi(bb)

print(f'success: {result.success}')
print(f'mean:    {result.mean:.3f}')
print(f'std:     {result.std:.3f}')
print(f'low_fraction: {result.low_fraction:.1%}')
print(f'\nSUMMARY STRING (this is what the agent sees):')
print(result.summary)

## One Schema in Detail

This JSON dict is what we pass to `tools=` in the API call. It's how the model knows what parameters to send.

In [ ]:
import json
ndvi_schema = next(s for s in TOOL_SCHEMAS if s['name'] == 'compute_ndvi')
print(json.dumps(ndvi_schema, indent=2))

## All 6 Tools

Call each tool on a small region and print its summary:

In [ ]:
bb_full = BoundingBox(0, 200, 0, 200)
bb_nw   = BoundingBox(0, 100, 0, 100)

for fn, args in [
    (tools.compute_ndvi,          {'region': bb_full}),
    (tools.compute_ndwi,          {'region': bb_nw}),
    (tools.compute_evi,           {'region': bb_full}),
    (tools.get_pixel_timeseries,  {'lat': 50, 'lon': 50, 'index': 'ndvi'}),
    (tools.flag_anomalous_regions,{'index': 'ndvi', 'threshold': 0.4, 'direction': 'below'}),
    (tools.compare_to_baseline,   {'region': bb_nw, 'index': 'ndvi'}),
]:
    result = fn(**args)
    print(f'=== {fn.__name__} ===')
    print(result.summary)
    print()

## Error Handling: What Happens With No Baseline?

This is **the key design decision**: tools return error results, they don't raise exceptions. This lets the agent reason about failures and adapt.

In [ ]:
load_scene('scene_b')  # Scene B has no baseline
result = tools.compare_to_baseline(BoundingBox(0, 200, 0, 200), index='ndvi')

print(f'success: {result.success}')
print(f'error:   {result.error_message}')
print('\n→ The agent receives this error message and can adapt.')

## How Tool Dispatch Works

The `dispatch_tool` function is just a dict lookup. No magic.

In [ ]:
from agent.loop import dispatch_tool
print(inspect.getsource(dispatch_tool))